# MEV — Máxima Extração de Valor na Rede Polygon
**Pesquisa FAPEMIG** | Coordenador: Prof. José Augusto Miranda Nacif | Bolsista: Aline Cristina Santos Silva

---

Este notebook documenta a **coleta e análise de dados on-chain** da rede Polygon (Layer-2),
com foco na medição do *Reordering Slippage* em swaps da Uniswap V3.

### Objetivo
Coletar eventos de Swap reais da blockchain para, posteriormente, calcular o slippage de reordenação
e comparar com a linha de base da Ethereum Mainnet — validando a hipótese **H** da proposta:
> *Contratos inteligentes em redes L2 apresentam um reordering slippage significativamente menor do que na rede principal Ethereum.*

### Perguntas de Pesquisa endereçadas
- **RQ1:** Qual a magnitude da diferença no Reordering Slippage médio entre Ethereum Mainnet e Polygon?
- **RQ2:** Qual a proporção entre Slippage Adversário (MEV) e Slippage de Colisão (benigno)?
- **RQ3:** Ativos voláteis (memecoins) apresentam maior slippage adversário também em L2?

---

## 1. Instalação de Dependências

Bibliotecas necessárias:
- **`web3`** — interface com nós da blockchain via RPC
- **`pandas`** — manipulação e análise dos dados coletados
- **`python-dotenv`** — carrega variáveis de ambiente do arquivo `.env` (protege a API key)
- **`matplotlib` / `seaborn`** — visualizações

In [2]:
# Execute uma vez para instalar as dependências
%pip install web3 pandas python-dotenv matplotlib seaborn --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Configuração — Carregando Credenciais

A API key da Alchemy fica armazenada no arquivo **`.env`** (nunca versionado no GitHub).
O arquivo `.env.example` no repositório documenta quais variáveis são necessárias sem expor os valores reais.



## 3. Conexão com a Rede Polygon via Alchemy

A conexão é feita via **RPC (Remote Procedure Call)** — um protocolo que permite consultar
o estado da blockchain sem precisar baixar todos os dados localmente.

A **Alchemy** atua como provedor de nó, fornecendo acesso à Polygon Mainnet de forma confiável e escalável.

**Por que Polygon?**
Por ser uma rede L2, espera-se que o *reordering slippage* seja menor do que na Ethereum Mainnet —
essa é exatamente a hipótese que esta pesquisa busca validar empiricamente.

In [3]:
import os
import requests
from dotenv import load_dotenv
from web3 import Web3
from web3.middleware import ExtraDataToPOAMiddleware

load_dotenv(override=True)
API_KEY = os.getenv("ALCHEMY_API_KEY")
RPC_URL = f"https://polygon-mainnet.g.alchemy.com/v2/{API_KEY}"

# Sessão customizada sem verificação SSL
session = requests.Session()
session.verify = False

from web3.middleware import ExtraDataToPOAMiddleware
from requests.adapters import HTTPAdapter

w3 = Web3(Web3.HTTPProvider(RPC_URL, session=session))
w3.middleware_onion.inject(ExtraDataToPOAMiddleware, layer=0)

try:
    bloco = w3.eth.block_number
    print(f" Conectado! Bloco: {bloco:,}")
except Exception as e:
    print(f" Erro: {e}")

/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


 Conectado! Bloco: 87,766,543


## 4. Definição do Contrato — Pool Uniswap V3 (USDC/WETH)

Na Uniswap V3, cada par de tokens possui um **contrato de pool dedicado**.
Toda vez que um swap ocorre, o contrato emite um **evento `Swap`** — um log público e imutável
gravado na blockchain, contendo:

| Campo | Descrição | Relevância para a pesquisa |
|---|---|---|
| `sqrtPriceX96` | Preço de execução real codificado | Base para calcular o slippage |
| `amount0 / amount1` | Volume negociado de cada token | Tamanho do swap (impacto no preço) |
| `sender / recipient` | Endereços envolvidos | Identificar padrões de ataque sandwich |
| `tick` | Posição na curva de preço | Faixa de liquidez utilizada |
| `blockNumber` | Bloco em que ocorreu | Agrupar transações por bloco para cálculo do reordering slippage |

Começamos com o pool **USDC/WETH 0.05%** por ser um dos mais líquidos na Polygon —
ideal para calibrar a metodologia antes de analisar ativos voláteis (RQ3).

In [4]:
# Endereço do pool USDC/WETH 0.05% na Polygon (Uniswap V3)
POOL_ADDRESS = Web3.to_checksum_address("0x45dda9cb7c25131df268515131f647d726f50608")

# ABI mínimo — apenas o evento Swap (não precisamos do ABI completo)
POOL_ABI = [
    {
        "anonymous": False,
        "inputs": [
            {"indexed": True,  "name": "sender",       "type": "address"},
            {"indexed": True,  "name": "recipient",    "type": "address"},
            {"indexed": False, "name": "amount0",      "type": "int256"},
            {"indexed": False, "name": "amount1",      "type": "int256"},
            {"indexed": False, "name": "sqrtPriceX96", "type": "uint160"},
            {"indexed": False, "name": "liquidity",    "type": "uint128"},
            {"indexed": False, "name": "tick",         "type": "int24"}
        ],
        "name": "Swap",
        "type": "event"
    }
]

contrato = w3.eth.contract(address=POOL_ADDRESS, abi=POOL_ABI)
print(f" Contrato carregado: {POOL_ADDRESS}")
print(f" Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon")

 Contrato carregado: 0x45dDa9cb7c25131DF268515131f647d726f50608
 Pool: USDC/WETH 0.05% — Uniswap V3 na Polygon


## 5. Coleta de Eventos de Swap

Estamos usando o pool WMATIC/USDC ( o mais negociado da rede) e estamos usando a janela para 3.000 blocos. O resultado foi 441 swaps cim 53 blocos tendo 2 ou mais swaps, que é exatamente o mínimo que precisamos para calcular o Reordering Slippage. 

In [7]:
import time
import pandas as pd

POOL_ADDRESS_WMATIC = Web3.to_checksum_address("0xa374094527e1673a86de625aa59517c5de346d32")
contrato_wmatic = w3.eth.contract(address=POOL_ADDRESS_WMATIC, abi=POOL_ABI)

JANELA_BLOCOS = 10
TOTAL_BLOCOS  = 3000   # aumentamos aqui — vai levar ~15 min para rodar

bloco_fim    = w3.eth.block_number
bloco_inicio = bloco_fim - TOTAL_BLOCOS

print(f"Coletando {TOTAL_BLOCOS:,} blocos em lotes de {JANELA_BLOCOS}...")
print(f"Estimativa: ~{TOTAL_BLOCOS // JANELA_BLOCOS * 0.3 / 60:.0f} minutos")
print(f"Bloco {bloco_inicio:,} → {bloco_fim:,}")
print()

todos_eventos = []
erros = 0

for i, inicio in enumerate(range(bloco_inicio, bloco_fim, JANELA_BLOCOS)):
    fim = min(inicio + JANELA_BLOCOS - 1, bloco_fim)
    try:
        eventos = contrato_wmatic.events.Swap.get_logs(
            from_block=inicio,
            to_block=fim
        )
        todos_eventos.extend(eventos)
        # Mostra progresso a cada 50 lotes
        if i % 50 == 0:
            print(f"  Progresso: bloco {inicio:,} | {len(todos_eventos)} swaps acumulados")
    except Exception as e:
        erros += 1
    time.sleep(0.3)

print(f"\nTotal coletado : {len(todos_eventos)} swaps")
print(f"Erros          : {erros}")

# ── Diagnóstico ───────────────────────────────────────────────────────────────
if todos_eventos:
    _tmp = pd.DataFrame([{"bloco": e["blockNumber"]} for e in todos_eventos])
    _por_bloco = _tmp.groupby("bloco").size()
    uteis = (_por_bloco >= 2).sum()
    print(f"\nBlocos com 2+ swaps : {uteis}")
    print(f"Média por bloco     : {_por_bloco.mean():.2f}")
    print(f"Máximo num bloco    : {_por_bloco.max()}")
    
    if uteis < 30:
        print("\n⚠ Ainda poucos blocos úteis — considere aumentar TOTAL_BLOCOS para 5000")
    else:
        print("\n✓ Amostra suficiente para calcular o Reordering Slippage!")

/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Coletando 3,000 blocos em lotes de 10...
Estimativa: ~2 minutos
Bloco 87,763,662 → 87,766,662



/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  Progresso: bloco 87,763,662 | 3 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

...

  Progresso: bloco 87,765,662 | 277 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

  Progresso: bloco 87,766,162 | 386 swaps acumulados


/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/aline/slippage-analysis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'polygon-mainnet.g.alchemy.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv


Total coletado : 441 swaps
Erros          : 0

Blocos com 2+ swaps : 53
Média por bloco     : 1.20
Máximo num bloco    : 5

✓ Amostra suficiente para calcular o Reordering Slippage!


## 6. Estruturação dos Dados em DataFrame

Convertemos os eventos brutos da blockchain em um **DataFrame pandas** estruturado.
Cada linha representa um swap individual, com todos os campos necessários para
o cálculo do Reordering Slippage na próxima etapa.

In [13]:
import csv
import json
from pathlib import Path

def evento_para_dict(evento):
    """Converte um evento Web3 para um dicionário plano com todos os campos."""
    d = {}

    # ── Campos do topo ────────────────────────────────────────────────────────
    d["event"]            = evento.get("event")
    d["address"]          = evento.get("address")
    d["blockNumber"]      = evento.get("blockNumber")
    d["transactionIndex"] = evento.get("transactionIndex")
    d["logIndex"]         = evento.get("logIndex")

    # HexBytes → string legível
    tx_hash    = evento.get("transactionHash")
    block_hash = evento.get("blockHash")
    d["transactionHash"] = tx_hash.hex()    if tx_hash    else None
    d["blockHash"]       = block_hash.hex() if block_hash else None

    # ── Args (parâmetros do Swap) ─────────────────────────────────────────────
    args = evento.get("args", {})
    for chave, valor in args.items():
        if hasattr(valor, "hex"):                  # HexBytes
            valor = valor.hex()
        elif isinstance(valor, (dict, list)):       # estrutura aninhada
            valor = json.dumps(valor, default=str)
        d[f"args_{chave}"] = valor

    return d


# ── Gera as linhas ────────────────────────────────────────────────────────────
linhas = [evento_para_dict(e) for e in todos_eventos]

if not linhas:
    print("Nenhum evento para salvar.")
else:
    todas_colunas = list(dict.fromkeys(k for linha in linhas for k in linha))

    caminho_csv = Path("dataFrame/swaps.csv")
    with caminho_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=todas_colunas, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(linhas)

    print(f"{len(linhas)} swaps salvos em '{caminho_csv.resolve()}'")
    print(f"   Colunas ({len(todas_colunas)}): {todas_colunas}")

441 swaps salvos em '/home/aline/slippage-analysis/dataFrame/swaps.csv'
   Colunas (14): ['event', 'address', 'blockNumber', 'transactionIndex', 'logIndex', 'transactionHash', 'blockHash', 'args_sender', 'args_recipient', 'args_amount0', 'args_amount1', 'args_sqrtPriceX96', 'args_liquidity', 'args_tick']


## 7. Conversão do Preço (sqrtPriceX96 → Preço Legível)

O campo `sqrtPriceX96` armazena o preço em formato interno da Uniswap V3:
a **raiz quadrada do preço**, multiplicada por 2⁹⁶ (para evitar decimais na EVM).

Para obter o preço real USDC por WMATIC , aplicamos:

$$price = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times \frac{10^{decimais\_token0}}{10^{decimais\_token1}}$$

Para USDC (6 decimais) / WMATIC (18 decimais):

$$price_{USDC/WMATIC} = \left(\frac{sqrtPriceX96}{2^{96}}\right)^2 \times 10^{12}$$

 Este preço de execução real é a base para calcular o **Reordering Slippage** — a diferença entre o preço que o usuário obteve e o preço que obteria em uma ordem aleatória de transações.

In [17]:
import pandas as pd

# ── Carrega os dados do novo pool (WMATIC/USDC) ───────────────────────────────
df = pd.read_csv("dataFrame/swaps.csv")

df = df.rename(columns={
    "blockNumber":       "bloco",
    "transactionIndex":  "tx_index",
    "args_sqrtPriceX96": "sqrtPriceX96",
    "args_amount0":      "WMATIC",
    "args_amount1":      "USDC",
    "args_sender":       "sender",
    "args_recipient":    "recipient",
})

df["sqrtPriceX96"] = pd.to_numeric(df["sqrtPriceX96"])
df["amount0"]      = pd.to_numeric(df["WMATIC"])
df["amount1"]      = pd.to_numeric(df["USDC"])

# ── Conversão de preço — pool WMATIC/USDC ────────────────────────────────────
# token0 = WMATIC (18 decimais)
# token1 = USDC   (6 decimais)
# sqrtPriceX96 dá sqrt(token1/token0) em unidades brutas
# Ajuste decimal: 10^(decimais_token1 - decimais_token0) = 10^(6-18) = 10^-12

Q96 = 2 ** 96

def sqrt_price_to_price_wmatic(sqrt_price_x96):
    """
    Converte sqrtPriceX96 → preço USDC por WMATIC.
    
    Passo a passo:
    1. Divide por 2^96 para desfazer o encoding da Uniswap
    2. Eleva ao quadrado para desfazer a raiz quadrada
    3. Multiplica por 10^(6-18) para corrigir as casas decimais
       (USDC tem 6, WMATIC tem 18 — diferença de 12 casas)
    """
    preco_bruto = (sqrt_price_x96 / Q96) ** 2   # ainda em unidades brutas
    return preco_bruto * 1e12                     # corrige os 12 decimais

df["preco_execucao"] = df["sqrtPriceX96"].apply(sqrt_price_to_price_wmatic)

# ── Resultado ─────────────────────────────────────────────────────────────────
print("Preços calculados para o pool WMATIC/USDC!")
print(f"  Preço médio : ${df['preco_execucao'].mean():.4f} USDC/WMATIC")
print(f"  Mínimo      : ${df['preco_execucao'].min():.4f}")
print(f"  Máximo      : ${df['preco_execucao'].max():.4f}")
print(f"  Total swaps : {len(df)}")
print()

# ── Diagnóstico dos blocos úteis ──────────────────────────────────────────────
swaps_por_bloco = df.groupby("bloco").size()
blocos_uteis    = swaps_por_bloco[swaps_por_bloco >= 2]

print(f"Blocos com 2+ swaps : {len(blocos_uteis)}")
print(f"Swaps nesses blocos : {blocos_uteis.sum()}")
print()

# ── Tabela dos primeiros swaps ────────────────────────────────────────────────
df[["bloco", "tx_index", "preco_execucao", "WMATIC", "USDC"]].head(10)

Preços calculados para o pool WMATIC/USDC!
  Preço médio : $0.0926 USDC/WMATIC
  Mínimo      : $0.0919
  Máximo      : $0.0934
  Total swaps : 441

Blocos com 2+ swaps : 53
Swaps nesses blocos : 128



,bloco,tx_index,preco_execucao,WMATIC,USDC
0,87763669,173,0.091874,518955037126902218752,-47674908
1,87763670,201,0.091922,-316258367811808905259,29077955
2,87763671,176,0.091916,36318778563304050688,-3336715
3,87763675,149,0.091925,-59695499375724358541,5490000
4,87763677,4,0.091934,-54291851046807148804,4993515
5,87763688,143,0.091936,-18313300018856565000,1684475
6,87763694,14,0.091938,-8124851453877147922,747348
7,87763695,181,0.091938,-46651225359598526,4292
8,87763696,18,0.091932,38680311440502637524,-3554282
9,87763707,2,0.092023,-598454950413382391278,55071750


Quando WMATIC > 0 e USDC < 0 significa que o pool recebeu WMATIC e entregou USDC. Quando WMATIC<0 e USDC>0 significa que o pool recebeu USDC e entregou WMATIC. 